# Kaggle Word2Vec NLP Tutorial Part 3：More Fun With Word Vectors

Part 2 训练出了每个词的向量。Part 3 要把词向量用于情感分类。

关键问题是：Kaggle 要判断的是整篇影评的情感，而 Word2Vec 给的是单个词的向量，所以需要把词向量聚合成影评级别的特征。


## 1. 导入库


In [ ]:
import csv
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans


## 2. 查找和读取数据


In [ ]:
def show_available_inputs():
    input_root = Path('/kaggle/input')
    if input_root.exists():
        print('Available /kaggle/input entries:')
        for path in sorted(input_root.glob('*')):
            print(' -', path)
            for child in sorted(path.glob('*'))[:10]:
                print('    -', child.name)
    else:
        print('/kaggle/input does not exist. If running locally, put files under data/.')

def find_file(filename, search_dirs, required=True):
    for directory in search_dirs:
        directory = Path(directory)
        direct_path = directory / filename
        zipped_path = directory / f'{filename}.zip'
        if direct_path.exists():
            return direct_path
        if zipped_path.exists():
            return zipped_path
        if directory.exists():
            matches = list(directory.rglob(filename))
            if matches:
                return matches[0]
            zipped_matches = list(directory.rglob(f'{filename}.zip'))
            if zipped_matches:
                return zipped_matches[0]
    if required:
        show_available_inputs()
        raise FileNotFoundError(
            f'Cannot find {filename}. In Kaggle, click Add Input / Add data and attach the word2vec-nlp-tutorial competition dataset.'
        )
    return None

def read_tsv(path):
    path = Path(path)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            tsv_names = [name for name in archive.namelist() if name.endswith('.tsv')]
            if not tsv_names:
                raise ValueError(f'No TSV file found inside {path}')
            with archive.open(tsv_names[0]) as file_obj:
                return pd.read_csv(file_obj, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)
    return pd.read_csv(path, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)

search_dirs = [
    '/kaggle/input/word2vec-nlp-tutorial',
    '/kaggle/input',
    'data',
    '.',
]

train_path = find_file('labeledTrainData.tsv', search_dirs)
test_path = find_file('testData.tsv', search_dirs, required=False)
unlabeled_path = find_file('unlabeledTrainData.tsv', search_dirs)

print('Train file:    ', train_path)
print('Test file:     ', test_path)
print('Unlabeled file:', unlabeled_path)


train = read_tsv(train_path)
unlabeled_train = read_tsv(unlabeled_path)
test = read_tsv(test_path) if test_path is not None else None

print('Train shape:    ', train.shape)
print('Unlabeled shape:', unlabeled_train.shape)
if test is not None:
    print('Test shape:     ', test.shape)

display(train.head())


## 3. 文本清洗函数

这里分类时会去掉停用词，因为平均词向量时，停用词通常会增加噪音。


In [ ]:
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def review_to_wordlist(raw_review, remove_stopwords=False):
    text = BeautifulSoup(raw_review, 'html.parser').get_text()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.lower().split()
    if remove_stopwords:
        words = [word for word in words if word not in STOP_WORDS]
    return words

def simple_sentence_split(raw_review):
    return [sentence.strip() for sentence in re.split(r'[.!?]+', raw_review) if sentence.strip()]

try:
    import nltk.data
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    print('Using NLTK punkt sentence tokenizer.')
except Exception:
    tokenizer = None
    print('NLTK punkt tokenizer is not available. Falling back to simple regex sentence splitting.')

def review_to_sentences(raw_review, remove_stopwords=False):
    if tokenizer is not None:
        raw_sentences = tokenizer.tokenize(raw_review.strip())
    else:
        raw_sentences = simple_sentence_split(raw_review)

    sentences = []
    for raw_sentence in raw_sentences:
        words = review_to_wordlist(raw_sentence, remove_stopwords=remove_stopwords)
        if words:
            sentences.append(words)
    return sentences


## 4. 加载或训练 Word2Vec 模型

如果已经运行过 Part 2，这里会直接加载保存的模型。

如果找不到模型，这个 notebook 会自动重新训练一个，保证文件可以独立运行。


In [ ]:
MODEL_PATHS = [
    Path('/kaggle/working/300features_40minwords_10context.model'),
    Path('300features_40minwords_10context.model'),
]

model_path = next((path for path in MODEL_PATHS if path.exists()), None)

if model_path is not None:
    print('Loading Word2Vec model:', model_path)
    model = Word2Vec.load(str(model_path))
else:
    print('No saved Word2Vec model found. Training a new model first...')
    sentences = []
    for review in train['review']:
        sentences.extend(review_to_sentences(review, remove_stopwords=False))
    for review in unlabeled_train['review']:
        sentences.extend(review_to_sentences(review, remove_stopwords=False))

    model = Word2Vec(
        sentences=sentences,
        vector_size=300,
        min_count=40,
        workers=4,
        window=10,
        sample=1e-3,
        sg=0,
        seed=42,
    )
    output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    model_path = output_dir / '300features_40minwords_10context.model'
    model.save(str(model_path))
    print('Saved Word2Vec model:', model_path)

NUM_FEATURES = model.wv.vector_size
print('Vocabulary size:', len(model.wv.index_to_key))
print('Vector size:', NUM_FEATURES)


## 5. 方法一：平均词向量

一篇评论里有很多词，每个词都有一个 300 维向量。

最直接的做法是把这篇评论里所有词向量求平均，得到一个固定长度的影评向量。


In [ ]:
def make_feature_vec(words, model, num_features):
    feature_vec = np.zeros((num_features,), dtype='float32')
    n_words = 0
    vocab = set(model.wv.index_to_key)

    for word in words:
        if word in vocab:
            feature_vec += model.wv[word]
            n_words += 1

    if n_words > 0:
        feature_vec /= n_words

    return feature_vec

def get_avg_feature_vecs(reviews, model, num_features):
    review_vecs = np.zeros((len(reviews), num_features), dtype='float32')

    for i, review in enumerate(reviews):
        if i % 1000 == 0:
            print(f'Review {i} of {len(reviews)}')
        review_vecs[i] = make_feature_vec(review, model, num_features)

    return review_vecs


## 6. 生成平均词向量特征


In [ ]:
clean_train_reviews = [
    review_to_wordlist(review, remove_stopwords=True)
    for review in train['review']
]

train_data_vecs = get_avg_feature_vecs(clean_train_reviews, model, NUM_FEATURES)
print('Average vector train shape:', train_data_vecs.shape)

if test is not None:
    clean_test_reviews = [
        review_to_wordlist(review, remove_stopwords=True)
        for review in test['review']
    ]
    test_data_vecs = get_avg_feature_vecs(clean_test_reviews, model, NUM_FEATURES)
    print('Average vector test shape:', test_data_vecs.shape)


## 7. 用平均词向量训练随机森林并生成提交文件


In [ ]:
forest_avg = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

forest_avg.fit(train_data_vecs, train['sentiment'])
print('Average Word2Vec RandomForest trained.')

if test is not None:
    result_avg = forest_avg.predict(test_data_vecs)
    output_avg = pd.DataFrame({
        'id': test['id'],
        'sentiment': result_avg.astype(int),
    })

    output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    avg_output_path = output_dir / 'Word2Vec_AverageVectors.csv'
    output_avg.to_csv(avg_output_path, index=False, quoting=csv.QUOTE_NONE)
    print('Saved:', avg_output_path)
    display(output_avg.head())


## 8. 方法二：Bag of Centroids

平均词向量会丢失很多信息。教程还尝试了另一种方法：先把词向量聚类，然后把每篇评论表示成“词簇计数”。

这类似 Part 1 的 Bag-of-Words，但统计的不是单词，而是语义相近的一组词。

KMeans 聚类可能比较慢，所以默认先关闭。需要完整复现时，把 `RUN_BAG_OF_CENTROIDS` 改成 `True`。


In [ ]:
RUN_BAG_OF_CENTROIDS = False

if RUN_BAG_OF_CENTROIDS:
    word_vectors = model.wv.vectors
    num_clusters = max(2, word_vectors.shape[0] // 5)
    print('Number of clusters:', num_clusters)

    kmeans = KMeans(
        n_clusters=num_clusters,
        random_state=42,
        n_init='auto',
    )
    cluster_ids = kmeans.fit_predict(word_vectors)

    word_centroid_map = dict(zip(model.wv.index_to_key, cluster_ids))
    print('Created word to centroid map.')
else:
    print('Bag of Centroids section skipped. Set RUN_BAG_OF_CENTROIDS = True to run it.')


In [ ]:
def create_bag_of_centroids(wordlist, word_centroid_map):
    num_centroids = max(word_centroid_map.values()) + 1
    bag = np.zeros(num_centroids, dtype='float32')

    for word in wordlist:
        if word in word_centroid_map:
            bag[word_centroid_map[word]] += 1

    return bag

if RUN_BAG_OF_CENTROIDS:
    train_centroids = np.zeros((len(clean_train_reviews), num_clusters), dtype='float32')
    for i, review in enumerate(clean_train_reviews):
        if i % 1000 == 0:
            print(f'Train review {i} of {len(clean_train_reviews)}')
        train_centroids[i] = create_bag_of_centroids(review, word_centroid_map)

    forest_centroids = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    )
    forest_centroids.fit(train_centroids, train['sentiment'])

    if test is not None:
        test_centroids = np.zeros((len(clean_test_reviews), num_clusters), dtype='float32')
        for i, review in enumerate(clean_test_reviews):
            if i % 1000 == 0:
                print(f'Test review {i} of {len(clean_test_reviews)}')
            test_centroids[i] = create_bag_of_centroids(review, word_centroid_map)

        result_centroids = forest_centroids.predict(test_centroids)
        output_centroids = pd.DataFrame({
            'id': test['id'],
            'sentiment': result_centroids.astype(int),
        })
        output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
        centroids_output_path = output_dir / 'BagOfCentroids.csv'
        output_centroids.to_csv(centroids_output_path, index=False, quoting=csv.QUOTE_NONE)
        print('Saved:', centroids_output_path)
        display(output_centroids.head())


## 9. 小结

Part 3 的核心是从词向量变成影评向量。

平均词向量很简单，但会丢失词序和重点信息。

Bag of Centroids 利用了词向量聚类，但本质上仍然是计数方法，也会丢失词序。

这解释了为什么 Word2Vec 听起来更高级，但在这个教程里不一定明显超过 Part 1 的词袋模型。
